<a href="https://colab.research.google.com/github/jyj-08/Speech-Challenge-2026/blob/main/notebooks/run_on_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🚀 Speech Challenge 2026 - 구글 코랩(Colab) 원클릭 실행 템플릿

이 노트북은 **Speech Challenge 2026** (극한 전장 환경 음성 인식 모델 개발) 프로젝트를 Google Colab 환경에서 손쉽게 가져와 학습 및 평가할 수 있도록 구성된 전용 템플릿입니다.

Hugging Face `datasets` 라이브러리가 오디오 데이터를 내부 캐시 경로(`~/.cache/huggingface/datasets/`)로 자동 다운로드하고 관리하므로, **별도의 데이터 다운로드나 복사 과정 없이** 셀을 순서대로 실행하기만 하면 전체 파이프라인이 구동됩니다.

### 📌 실행 전 체크리스트
1. **GPU 런타임 활성화**: 상단 메뉴 `런타임` ➔ `런타임 유형 변경` ➔ 하드웨어 가속기를 **GPU (T4, V100, A100 등)**로 선택합니다.
2. **순차적 실행**: 위에서부터 셀을 하나씩 순서대로 `Shift + Enter`로 실행합니다.

## 1. 구글 드라이브 마운트 (Google Drive Mount)
학습 완료된 모델 가중치(체크포인트)와 평가 리포트를 세션 종료 후에도 영구 보존하기 위해 구글 드라이브를 연결합니다.

In [ ]:
# 구글 드라이브 마운트
from google.colab import drive
drive.mount('/content/drive')

## 2. 깃허브 리포지토리 클론 및 작업 디렉토리 이동
본인의 GitHub 리포지토리 코드를 Colab 인스턴스로 복제(`git clone`)하고, 프로젝트 루트 폴더로 진입합니다.

> 💡 **주의**: 아래 URL의 `[본인_깃허브_주소]` 부분을 본인의 GitHub 리포지토리 주소(예: `https://github.com/username/Speech-Challenge-2026.git`)로 수정해 주세요.

In [ ]:
# 깃허브 리포지토리 클론 (사용자 주소로 변경 필요)
# 예시: !git clone https://github.com/my-account/Speech-Challenge-2026.git
!git clone [본인_깃허브_주소]

# 프로젝트 디렉토리 내부로 이동
%cd Speech-Challenge-2026

# 현재 작업 경로 및 파일 확인
!pwd
!ls -la

In [ ]:
!git clone https://github.com/jyj-08/Speech-Challenge-2026.git
%cd Speech-Challenge-2026

## 3. GPU 환경 확인 및 패키지 의존성 설치
할당된 GPU 상태를 점검하고, 프로젝트 구동에 필요한 패키지(`torch`, `transformers`, `peft`, `audiomentations` 등)를 `requirements.txt`를 통해 일괄 설치합니다.

In [ ]:
# 1) GPU 할당 및 VRAM 상태 확인
!nvidia-smi

# 2) requirements.txt 패키지 일괄 설치
!pip install --upgrade pip
!pip install -r requirements.txt

## 4. 드라이 런 (Dry-Run: 데이터 자동 로드 및 파이프라인 빠른 사전 테스트)

> 💡 **데이터 자동 다운로드 안내**:
> 별도의 데이터 다운로드나 복사 과정이 필요 없습니다! 아래 스크립트를 실행하면 Hugging Face `datasets` 라이브러리가 지정된 음성 데이터셋(기본: LibriSpeech clean)을 코랩 환경의 캐시 디렉토리(`~/.cache/huggingface/datasets/`)로 **자동 다운로드하고 캐싱**합니다.

본격적인 대규모 학습에 앞서, **GPU Out-Of-Memory(OOM) 에러**나 **데이터 전처리/모델 텐서 차원 버그**가 없는지 5 step만 초경량으로 실행하여 정상 구동을 검증합니다.

* `--epochs 1`: 1 에포크로 제한
* `--batch_size 2`: 적은 배치 사이즈로 OOM 방지 및 빠른 초기화
* `--max_steps 5`: 5 스텝만 실행 후 즉시 종료하여 파이프라인 정상 동작 여부만 검증

In [ ]:
# 드라이 런 실행 (데이터 자동 다운로드 & 5스텝 사전 검증)
!python src/train.py \
    --config configs/default_config.yaml \
    --epochs 1 \
    --batch_size 2 \
    --max_steps 5 \
    --output_dir ./checkpoints/dry_run_test

## 5. 본 학습 (Main Training) 및 가중치 구글 드라이브 영구 저장
드라이 런 검증이 완료되면 전체 데이터를 대상으로 본격적인 LoRA 파인튜닝을 진행합니다.

체크포인트 저장 경로(`--output_dir`)를 **마운트된 구글 드라이브 경로**(`/content/drive/MyDrive/...`)로 직접 지정하여, 코랩 세션이 끊기거나 타임아웃이 발생해도 학습된 모델 가중치가 안전하게 영구 저장되도록 설정합니다.

In [ ]:
# 1) 구글 드라이브 내 체크포인트 저장 폴더 생성
!mkdir -p /content/drive/MyDrive/Speech-Challenge-2026/checkpoints/tactical_whisper

# 2) 본 학습 파이프라인 실행 (가중치는 구글 드라이브로 직접 저장)
!python src/train.py \
    --config configs/default_config.yaml \
    --epochs 3 \
    --batch_size 4 \
    --learning_rate 0.001 \
    --output_dir /content/drive/MyDrive/Speech-Challenge-2026/checkpoints/tactical_whisper

# 3) 구글 드라이브에 저장된 모델 가중치 파일 확인
!ls -lh /content/drive/MyDrive/Speech-Challenge-2026/checkpoints/tactical_whisper

## (선택) 학습 완료 모델 평가 (Evaluation)
구글 드라이브에 저장된 LoRA 체크포인트를 불러와 전장 열화 환경에서의 WER 및 CER 성능을 평가하고 결과를 JSON 리포트로 저장합니다.

In [ ]:
!python src/evaluate.py \
    --checkpoint_dir /content/drive/MyDrive/Speech-Challenge-2026/checkpoints/tactical_whisper \
    --metrics wer cer \
    --output_file /content/drive/MyDrive/Speech-Challenge-2026/checkpoints/eval_results.json